# Performance Metrics: Practice Exercise

Build a prompt strategy analyzer that tracks performance metrics across different prompting techniques on the same model.

**What you'll implement:**
- A method to track and record metrics for a single LLM call

**Estimated time:** 10-15 minutes

## Setup

Run this cell to import dependencies and initialize the API client.

In [11]:
!pip install langchain-core langchain-openai

In [12]:
import os

def load_api_key():
  IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


  if not IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv()
    # Verify OpenAI API key is set
    if not os.getenv("OPENAI_API_KEY"):
      ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    from google.colab import userdata
    OPENAI_API_KEY=userdata.get('OPEN_AI_API_KEY')
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [13]:
load_api_key()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

In [14]:
# Setup - run this cell first


import time
from datetime import datetime
from typing import Dict, List

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


# Initialize the model - we'll use gpt-4o to compare prompt strategies
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# Pricing for cost calculation (USD per 1K tokens)
PRICING = {
    "gpt-4o": {"input": 0.0025, "output": 0.01}
}

You're building a `PromptStrategyAnalyzer` that compares how different prompting techniques perform on the same task. Instead of comparing models, you'll compare prompt strategies:

- **Zero-shot**: Direct question with no examples
- **Few-shot**: Question with examples provided
- **Chain-of-thought**: Question asking for step-by-step reasoning

For each call, you need to track:
- `strategy`: The name of the prompting strategy used
- `latency_ms`: Time taken in milliseconds
- `input_tokens`: Number of prompt tokens
- `output_tokens`: Number of completion tokens  
- `total_cost_usd`: Cost calculated from token usage
- `success`: Whether the call succeeded
- `response`: The LLM's response content

The `generate_comparison_report` method is already implemented for you - you only need to implement `track_strategy`.

## Implement the Analyzer

Your task is to implement the track_strategy function below.

In [18]:
class PromptStrategyAnalyzer:
    """Analyze and compare performance metrics across different prompting strategies."""

    def __init__(self, llm: ChatOpenAI, pricing: Dict):
        """
        Initialize the analyzer.

        Args:
            llm: The LangChain ChatOpenAI instance to use for all calls
            pricing: Dict with model pricing, e.g. {"gpt-4o": {"input": 0.0025, "output": 0.01}}
        """
        self.llm = llm
        self.pricing = pricing
        self.results: List[Dict] = []

    # TODO: Implement this
    def track_strategy(self, strategy_name: str, prompt: str) -> Dict:
        """
        Execute a prompt and track its performance metrics.

        Args:
            strategy_name: Label for the prompting strategy (e.g., "zero-shot", "few-shot")
            prompt: The full prompt text to send to the LLM

        Returns:
            Dict containing:
                - strategy: The strategy name
                - latency_ms: Response time in milliseconds (rounded to 2 decimals)
                - input_tokens: Number of prompt tokens from response metadata
                - output_tokens: Number of completion tokens from response metadata
                - total_cost_usd: Calculated cost (rounded to 6 decimals)
                - success: True if call succeeded, False otherwise
                - response: The response content (or error message if failed)

        Notes:
            - Use time.time() to measure latency
            - Token usage is in response.response_metadata['token_usage']
            - Cost formula: (input_tokens/1000 * input_price) + (output_tokens/1000 * output_price)
            - Get model name from self.llm.model_name
            - Append the result dict to self.results before returning
        """
        start_time = time.time()
        try:
            response = self.llm.invoke([HumanMessage(content=prompt)])
            end_time = time.time()
            latency_ms = round((end_time - start_time) * 1000, 2)
            usage_metadata = response.response_metadata.get('token_usage', {})
            input_tokens = usage_metadata.get('prompt_tokens', 0)
            output_tokens = usage_metadata.get('completion_tokens', 0)
            model_name = self.llm.model_name
            pricing = self.pricing.get(model_name,self.pricing["gpt-4o"])
            input_price = (input_tokens/1000) * pricing.get('input', 0)
            output_price = (output_tokens/1000) * pricing.get('output', 0)
            total_cost_usd = round(input_price + output_price, 6)
            metric = {
                "strategy":strategy_name,
                "latency_ms":latency_ms,
                "input_tokens":input_tokens,
                "output_tokens":output_tokens,
                "total_cost_usd":total_cost_usd,
                "success":True,
                "response":response.content
            }

        except Exception as ex:
          end_time = time.time()
          latency_ms = round((end_time - start_time) * 1000, 2)
          metric = {
                "strategy":strategy_name,
                "latency_ms":latency_ms,
                "input_tokens":0,
                "output_tokens":0,
                "total_cost_usd": 0,
                "success":False,
                "response":str(ex)
            }
        self.results.append(metric)
        return metric


    def generate_comparison_report(self) -> str:
        """Generate a formatted comparison report of all tracked strategies."""
        successful = [r for r in self.results if r["success"]]

        if not successful:
            return "No successful results to report."

        lines = []
        lines.append("PROMPT STRATEGY COMPARISON")
        lines.append("=" * 70)
        lines.append(f"{'Strategy':<20} {'Latency (ms)':<15} {'Input Tokens':<15} {'Output Tokens':<15} {'Cost ($)':<12}")
        lines.append("-" * 70)

        for r in successful:
            lines.append(
                f"{r['strategy']:<20} {r['latency_ms']:<15.2f} {r['input_tokens']:<15} {r['output_tokens']:<15} ${r['total_cost_usd']:.6f}"
            )

        lines.append("=" * 70)

        # Find fastest and cheapest
        fastest = min(successful, key=lambda x: x["latency_ms"])
        cheapest = min(successful, key=lambda x: x["total_cost_usd"])

        lines.append(f"Fastest: {fastest['strategy']} ({fastest['latency_ms']:.2f} ms)")
        lines.append(f"Cheapest: {cheapest['strategy']} (${cheapest['total_cost_usd']:.6f})")

        return "\n".join(lines)

## Run Your Implementation

Test your analyzer by comparing three prompting strategies on a sentiment classification task.

In [16]:
# Test prompts for sentiment classification
text_to_classify = "The movie started slow but the ending was absolutely incredible. I couldn't stop thinking about it for days."

zero_shot_prompt = f"""Classify the sentiment of this text as positive, negative, or neutral.

Text: {text_to_classify}

Sentiment:"""

few_shot_prompt = f"""Classify the sentiment of texts as positive, negative, or neutral.

Text: "I love this product! Best purchase ever."
Sentiment: positive

Text: "Terrible experience. Would not recommend."
Sentiment: negative

Text: "It's okay, nothing special."
Sentiment: neutral

Text: "{text_to_classify}"
Sentiment:"""

chain_of_thought_prompt = f"""Classify the sentiment of this text as positive, negative, or neutral.

Text: {text_to_classify}

Think through this step by step:
1. Identify the key emotional words and phrases
2. Consider the overall tone
3. Weigh positive vs negative elements
4. Provide your final classification

Analysis:"""

In [19]:
# Run your implementation
analyzer = PromptStrategyAnalyzer(llm, PRICING)

# Track each strategy
print("Testing zero-shot...")
analyzer.track_strategy("zero-shot", zero_shot_prompt)

print("Testing few-shot...")
analyzer.track_strategy("few-shot", few_shot_prompt)

print("Testing chain-of-thought...")
analyzer.track_strategy("chain-of-thought", chain_of_thought_prompt)

# Generate and print the comparison report
print("\n")
print(analyzer.generate_comparison_report())

# Print response snippets
print("\n\nRESPONSE SNIPPETS")
print("=" * 70)
for r in analyzer.results:
    if r["success"]:
        snippet = r["response"][:100] + "..." if len(r["response"]) > 100 else r["response"]
        print(f"\n{r['strategy'].upper()}:")
        print(f"  {snippet}")

Testing zero-shot...
Testing few-shot...
Testing chain-of-thought...


PROMPT STRATEGY COMPARISON
Strategy             Latency (ms)    Input Tokens    Output Tokens   Cost ($)    
----------------------------------------------------------------------
zero-shot            505.58          47              1               $0.000128
few-shot             477.52          94              1               $0.000245
chain-of-thought     2585.08         86              227             $0.002485
Fastest: few-shot (477.52 ms)
Cheapest: zero-shot ($0.000128)


RESPONSE SNIPPETS

ZERO-SHOT:
  Positive

FEW-SHOT:
  positive

CHAIN-OF-THOUGHT:
  1. Identify the key emotional words and phrases:
   - "started slow" - This phrase suggests a potent...
